# Decision Trees from scratch 
Sections: impurity measures -> information gain -> best split -> DecisionTree
classifier -> regression tree (variance reduction) -> pre-pruning ->
cost-complexity post-pruning -> Play Tennis ID3 worked example.

In [ ]:
import os, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(0)

## 1. Gini impurity and entropy
Gini(S) = 1 - sum p_k^2, Entropy(S) = -sum p_k log2 p_k.

In [ ]:
def gini_impurity(labels):
    """Gini impurity of a label array."""
    n = len(labels)
    if n == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    p = counts / n
    return float(1.0 - np.sum(p ** 2))

def entropy(labels):
    """Shannon entropy (bits) of a label array."""
    n = len(labels)
    if n == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    p = counts / n
    return float(-np.sum(p * np.log2(p)))

def variance(y):
    """Variance (regression impurity)."""
    return float(np.var(y)) if len(y) else 0.0

def information_gain(parent, left, right, criterion="gini"):
    """IG = impurity(parent) - weighted impurity(children). Also VR for 'variance'."""
    measure = {"gini": gini_impurity, "entropy": entropy, "variance": variance}[criterion]
    n, nl, nr = len(parent), len(left), len(right)
    if nl == 0 or nr == 0:
        return 0.0
    return measure(parent) - (nl / n) * measure(left) - (nr / n) * measure(right)

cats_dogs = np.array([0] * 6 + [1] * 4)
print("6 cats/4 dogs  gini=%.3f (doc 0.48)  entropy=%.3f (doc 0.971)"
      % (gini_impurity(cats_dogs), entropy(cats_dogs)))

# plot impurity curves for binary problem
p = np.linspace(0.001, 0.999, 200)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(p, 1 - p**2 - (1-p)**2, label='Gini')
ax.plot(p, -(p*np.log2(p) + (1-p)*np.log2(1-p)), label='Entropy')
ax.set_xlabel('p (class 1)'); ax.set_ylabel('impurity'); ax.legend(); ax.set_title('Impurity measures')
fig.tight_layout(); fig.savefig('plots/01_impurity_curves.png', dpi=120); plt.close(fig)

## 2-3. Best split search and the DecisionTree class
For each feature: sort, evaluate midpoints between consecutive distinct values,
using cumulative class counts so the search is O(n log n) per feature.
The same class handles regression (`task='regression'`, variance reduction, mean leaves).

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None,
                 value=None, n=0, impurity=0.0, y_stats=None):
        self.feature, self.threshold, self.left, self.right = feature, threshold, left, right
        self.value, self.n, self.impurity = value, n, impurity

    @property
    def is_leaf(self):
        return self.left is None


class DecisionTree:
    """CART-style tree (binary splits on numeric features).

    task: 'classification' (gini/entropy) or 'regression' (variance reduction).
    Pre-pruning: max_depth, min_samples_split, min_samples_leaf, min_gain.
    """
    def __init__(self, task="classification", criterion="gini", max_depth=None,
                 min_samples_split=2, min_samples_leaf=1, min_gain=0.0,
                 max_features=None, rng=None):
        self.task = task
        self.criterion = criterion if task == "classification" else "variance"
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_gain = min_gain
        self.max_features = max_features   # int or None (used by random forest)
        self.rng = rng if rng is not None else np.random.default_rng(0)

    # ---- impurity helpers
    def _impurity(self, y):
        return {"gini": gini_impurity, "entropy": entropy, "variance": variance}[self.criterion](y)

    def _leaf_value(self, y):
        if self.task == "regression":
            return float(np.mean(y))
        return int(np.argmax(np.bincount(y, minlength=self.n_classes_)))

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y)
        if self.task == "classification":
            y = y.astype(int)
            self.n_classes_ = int(y.max()) + 1
        self.n_features_ = X.shape[1]
        self.feature_importances_ = np.zeros(self.n_features_)
        self.root = self._build(X, y, depth=0)
        s = self.feature_importances_.sum()
        if s > 0:
            self.feature_importances_ /= s
        return self

    def _best_split(self, X, y):
        """Return (feature, threshold, gain) maximizing information gain."""
        n, d = X.shape
        parent_imp = self._impurity(y)
        feats = np.arange(d)
        if self.max_features is not None and self.max_features < d:
            feats = self.rng.choice(d, self.max_features, replace=False)
        best = (None, None, 0.0)
        for j in feats:
            order = np.argsort(X[:, j], kind="stable")
            xs, ys = X[order, j], y[order]
            if self.task == "classification":
                onehot = np.eye(self.n_classes_)[ys]
                cl = np.cumsum(onehot, axis=0)[:-1]          # counts left for split after i
                tot = onehot.sum(axis=0)
                cr = tot - cl
                nl = np.arange(1, n)[:, None]; nr = n - nl
                pl, pr = cl / nl, cr / nr
                if self.criterion == "gini":
                    il = 1 - (pl ** 2).sum(1); ir = 1 - (pr ** 2).sum(1)
                else:
                    with np.errstate(divide="ignore", invalid="ignore"):
                        il = -np.nansum(np.where(pl > 0, pl * np.log2(pl), 0), 1)
                        ir = -np.nansum(np.where(pr > 0, pr * np.log2(pr), 0), 1)
                nl, nr = nl[:, 0], nr[:, 0]
            else:
                c1 = np.cumsum(ys)[:-1]; c2 = np.cumsum(ys ** 2)[:-1]
                nl = np.arange(1, n); nr = n - nl
                s1, s2 = ys.sum() - c1, (ys ** 2).sum() - c2
                il = c2 / nl - (c1 / nl) ** 2
                ir = s2 / nr - (s1 / nr) ** 2
            gain = parent_imp - (nl / n) * il - (nr / n) * ir
            valid = (xs[1:] != xs[:-1]) & (nl >= self.min_samples_leaf) & (nr >= self.min_samples_leaf)
            if not valid.any():
                continue
            gain = np.where(valid, gain, -np.inf)
            i = int(np.argmax(gain))
            if gain[i] > best[2] + 1e-12:
                best = (int(j), float((xs[i] + xs[i + 1]) / 2), float(gain[i]))
        return best

    def _split_data(self, X, y, feature, threshold):
        mask = X[:, feature] <= threshold
        return X[mask], y[mask], X[~mask], y[~mask]

    def _build(self, X, y, depth):
        n = len(y)
        imp = self._impurity(y)
        node = Node(value=self._leaf_value(y), n=n, impurity=imp)
        if (imp == 0 or n < self.min_samples_split or
                (self.max_depth is not None and depth >= self.max_depth)):
            return node
        f, t, gain = self._best_split(X, y)
        if f is None or gain <= self.min_gain:
            return node
        Xl, yl, Xr, yr = self._split_data(X, y, f, t)
        self.feature_importances_[f] += n * gain      # weighted impurity decrease
        node.feature, node.threshold = f, t
        node.left = self._build(Xl, yl, depth + 1)
        node.right = self._build(Xr, yr, depth + 1)
        return node

    def _predict_one(self, x, node=None):
        node = node or self.root
        while not node.is_leaf:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node.value

    def predict(self, X):
        return np.array([self._predict_one(x) for x in np.asarray(X, float)])

    # ---- tree stats
    def n_leaves(self, node=None):
        node = node or self.root
        return 1 if node.is_leaf else self.n_leaves(node.left) + self.n_leaves(node.right)

    def depth(self, node=None):
        node = node or self.root
        return 0 if node.is_leaf else 1 + max(self.depth(node.left), self.depth(node.right))

    def print_tree(self, names=None, node=None, indent=""):
        node = node or self.root
        if node.is_leaf:
            print(indent + f"-> {node.value if self.task=='classification' else round(node.value,3)} (n={node.n})")
            return
        nm = names[node.feature] if names else f"x[{node.feature}]"
        print(indent + f"{nm} <= {node.threshold:.3f}  (n={node.n}, imp={node.impurity:.3f})")
        self.print_tree(names, node.left, indent + "  |  ")
        print(indent + "else")
        self.print_tree(names, node.right, indent + "  |  ")

def accuracy(a, b): return float(np.mean(a == b))

## 4. Synthetic 3-class data and decision boundaries (max_depth 2 vs 10)

In [ ]:
def make_blobs3(n=300, seed=1, spread=1.1):
    r = np.random.default_rng(seed)
    centers = np.array([[0, 0], [3, 3], [0, 4]])
    y = r.integers(0, 3, n)
    X = centers[y] + r.normal(0, spread, (n, 2))
    return X, y

def train_test_split(X, y, frac=0.3, seed=0):
    r = np.random.default_rng(seed)
    idx = r.permutation(len(y)); k = int(len(y) * frac)
    return X[idx[k:]], X[idx[:k]], y[idx[k:]], y[idx[:k]]

X, y = make_blobs3(600)
Xtr, Xte, ytr, yte = train_test_split(X, y)

def plot_boundary(ax, model, X, y, title):
    gx, gy = np.meshgrid(np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 200),
                         np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 200))
    Z = model.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, Z, alpha=0.25, cmap='viridis', levels=np.arange(-0.5, 3))
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=10, edgecolor='k', linewidth=0.2)
    ax.set_title(title)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, d in zip(axs, [2, 10]):
    t = DecisionTree(max_depth=d).fit(Xtr, ytr)
    plot_boundary(ax, t, Xtr, ytr, f"max_depth={d} train={accuracy(t.predict(Xtr),ytr):.2f} test={accuracy(t.predict(Xte),yte):.2f}")
    print(f"depth={d}: leaves={t.n_leaves()}, train={accuracy(t.predict(Xtr),ytr):.3f}, test={accuracy(t.predict(Xte),yte):.3f}")
fig.tight_layout(); fig.savefig('plots/02_boundaries_depth.png', dpi=120); plt.close(fig)

## 5. Gini vs entropy
They give nearly identical trees because both are smooth, concave, maximised at uniform
class distribution and zero at pure nodes, so they rank most splits the same way.

In [ ]:
for crit in ["gini", "entropy"]:
    accs, deps = [], []
    for s in range(5):
        Xa, ya = make_blobs3(400, seed=10 + s)
        a, b, c, d_ = train_test_split(Xa, ya, seed=s)
        t = DecisionTree(criterion=crit, max_depth=6).fit(a, c)
        accs.append(accuracy(t.predict(b), d_)); deps.append(t.depth())
    print(f"{crit:8s} mean test acc={np.mean(accs):.3f}  mean depth={np.mean(deps):.1f}")

## 6. Regression tree (variance reduction): y = sin(x) + noise

In [ ]:
xr = np.sort(rng.uniform(0, 2 * np.pi, 200))
yr = np.sin(xr) + rng.normal(0, 0.25, 200)
grid = np.linspace(0, 2 * np.pi, 400)
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, d in zip(axs, [1, 3, 10]):
    rt = DecisionTree(task="regression", max_depth=d).fit(xr[:, None], yr)
    ax.scatter(xr, yr, s=8, c='gray')
    ax.plot(grid, np.sin(grid), 'g--', label='true sin')
    ax.plot(grid, rt.predict(grid[:, None]), 'r', label='tree')
    mse = np.mean((rt.predict(xr[:, None]) - yr) ** 2)
    ax.set_title(f"regression depth={d}, train MSE={mse:.3f}"); ax.legend()
fig.tight_layout(); fig.savefig('plots/03_regression_tree.png', dpi=120); plt.close(fig)

## 7. Pre-pruning: max_depth, min_samples_leaf, min_gain

In [ ]:
Xn, yn = make_blobs3(800, seed=5, spread=1.6)
a, b, c, d_ = train_test_split(Xn, yn, seed=3)
depths = list(range(1, 15))
tr = [accuracy(DecisionTree(max_depth=k).fit(a, c).predict(a), c) for k in depths]
te = [accuracy(DecisionTree(max_depth=k).fit(a, c).predict(b), d_) for k in depths]
leafs = [1, 2, 5, 10, 20, 40]
te_leaf = [accuracy(DecisionTree(min_samples_leaf=m).fit(a, c).predict(b), d_) for m in leafs]
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(depths, tr, 'o-', label='train'); axs[0].plot(depths, te, 's-', label='test')
axs[0].set_xlabel('max_depth'); axs[0].set_ylabel('accuracy'); axs[0].legend()
axs[1].plot(leafs, te_leaf, 'o-'); axs[1].set_xlabel('min_samples_leaf'); axs[1].set_ylabel('test accuracy')
fig.tight_layout(); fig.savefig('plots/04_prepruning.png', dpi=120); plt.close(fig)
print("best depth by test acc:", depths[int(np.argmax(te))], " full-tree test acc:", te[-1])

## 8. Post-pruning: cost-complexity (weakest-link) pruning
R_alpha(T) = R(T) + alpha * |leaves|. Repeatedly collapse the internal node with the smallest
g(t) = (R(t) - R(T_t)) / (|leaves(T_t)| - 1), producing a nested sequence of trees;
then choose alpha on a validation set.

In [ ]:
import copy

def _node_error(node):            # misclassifications if node were a leaf (count-based, needs stored counts)
    return node.err

def annotate_errors(tree, X, y):
    """Store on each node the number of training errors if it were collapsed to a leaf."""
    def rec(node, idx):
        yy = y[idx]
        node.err = float(np.sum(yy != node.value)) if tree.task == "classification" \
            else float(np.sum((yy - node.value) ** 2))
        if not node.is_leaf:
            m = X[idx, node.feature] <= node.threshold
            rec(node.left, idx[m]); rec(node.right, idx[~m])
    rec(tree.root, np.arange(len(y)))

def subtree_stats(node):
    if node.is_leaf:
        return node.err, 1
    el, ll = subtree_stats(node.left); er, lr = subtree_stats(node.right)
    return el + er, ll + lr

def internal_nodes(node):
    if node.is_leaf:
        return []
    return [node] + internal_nodes(node.left) + internal_nodes(node.right)

def cost_complexity_path(tree, X, y):
    """Return list of (alpha, pruned_tree_copy) for the nested weakest-link sequence."""
    t = copy.deepcopy(tree)
    annotate_errors(t, X, y)
    seq = [(0.0, copy.deepcopy(t))]
    while not t.root.is_leaf:
        best_g, best_node = np.inf, None
        for nd in internal_nodes(t.root):
            e, l = subtree_stats(nd)
            g = (nd.err - e) / (l - 1)
            if g < best_g:
                best_g, best_node = g, nd
        best_node.left = best_node.right = None      # collapse
        seq.append((best_g, copy.deepcopy(t)))
    return seq

full = DecisionTree(max_depth=None).fit(a, c)
path = cost_complexity_path(full, a, c)
val_acc = [accuracy(t.predict(b), d_) for _, t in path]
sizes = [t.n_leaves() for _, t in path]
k = int(np.argmax(val_acc))
print(f"full tree: {sizes[0]} leaves, test acc {val_acc[0]:.3f}")
print(f"best pruned: alpha={path[k][0]:.2f}, {sizes[k]} leaves, test acc {val_acc[k]:.3f}")
print(f"pre-pruned best depth: acc {max(te):.3f}")
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(sizes, val_acc, 'o-'); ax.invert_xaxis()
ax.set_xlabel('number of leaves (more pruning ->)'); ax.set_ylabel('test accuracy'); ax.set_title('Cost-complexity pruning')
fig.tight_layout(); fig.savefig('plots/05_cost_complexity.png', dpi=120); plt.close(fig)

## 9. Feature importance (impurity decrease)

In [ ]:
Xf = rng.normal(size=(500, 5)); yf = ((Xf[:, 0] + 0.5 * Xf[:, 1]) > 0).astype(int)
tf = DecisionTree(max_depth=5).fit(Xf, yf)
print("feature importances:", np.round(tf.feature_importances_, 3))
fig, ax = plt.subplots(figsize=(5, 3)); ax.bar(range(5), tf.feature_importances_)
ax.set_xlabel('feature'); ax.set_title('Importances'); fig.tight_layout()
fig.savefig('plots/06_feature_importance.png', dpi=120); plt.close(fig)

## 10. Play Tennis: ID3 (multiway categorical splits, entropy)
Reproduces Math/math.md: root entropy 0.940, IG(Outlook)=0.247 etc.

In [ ]:
names = ["Outlook", "Temperature", "Humidity", "Windy"]
rows = """Sunny Hot High False No
Sunny Hot High True No
Overcast Hot High False Yes
Rain Mild High False Yes
Rain Cool Normal False Yes
Rain Cool Normal True No
Overcast Cool Normal True Yes
Sunny Mild High False No
Sunny Cool Normal False Yes
Rain Mild Normal False Yes
Sunny Mild Normal True Yes
Overcast Mild High True Yes
Overcast Hot Normal False Yes
Rain Mild High True No""".split("\n")
data = [r.split() for r in rows]
XT = np.array([r[:4] for r in data]); yT = np.array([r[4] for r in data])

def ig_categorical(col, y):
    h = entropy(y)
    for v in np.unique(col):
        m = col == v
        h -= m.mean() * entropy(y[m])
    return h

print(f"Root entropy = {entropy(yT):.3f}, Gini = {gini_impurity(yT):.3f}")
for j, nm in enumerate(names):
    print(f"IG({nm}) = {ig_categorical(XT[:, j], yT):.3f}")

def id3(X, y, feats, indent=""):
    if len(np.unique(y)) == 1:
        print(indent + f"-> {y[0]}"); return y[0]
    if not feats:
        lab = max(set(y), key=list(y).count); print(indent + f"-> {lab}"); return lab
    gains = {j: ig_categorical(X[:, j], y) for j in feats}
    j = max(gains, key=gains.get)
    tree = {}
    for v in np.unique(X[:, j]):
        print(indent + f"{names[j]} = {v}  (IG={gains[j]:.3f})")
        m = X[:, j] == v
        tree[v] = id3(X[m], y[m], [f for f in feats if f != j], indent + "    ")
    return (j, tree)

tree_t = id3(XT, yT, [0, 1, 2, 3])

def id3_predict(tree, x):
    while isinstance(tree, tuple):
        tree = tree[1][x[tree[0]]]
    return tree
Q = ["Sunny", "Cool", "High", "False"]
print("Prediction for Q:", id3_predict(tree_t, Q), "(doc: No)")